# Deep Learning · PR 4 — **Project A: CNN from Scratch (CIFAR-10 Image Classifier)**

**Declaration: Project A — CNN from Scratch** · Framework: TensorFlow 2.x / Keras · Dataset: CIFAR-10

**Pipeline in this notebook**

| Task | Topic |
|---|---|
| A1 | Data loading, EDA, normalisation, one-hot encoding, CNN vs ANN |
| A2 | Convolution operation, padding (VALID/SAME), strides, feature maps |
| A3 | Pooling, Flatten, LeNet-5 (1998), modern custom CNN |
| A4 | Backprop in CNN, training with callbacks, curves, confusion matrix, errors |
| A5 | Data augmentation (ImageDataGenerator + Keras layers) |
| A6 | Transfer learning (MobileNetV2 feature extraction + fine-tuning), results table, residual block |

> **Tip:** set `FAST_MODE = True` in the config cell for a quick smoke test (a few epochs). Use `False` for the final *Restart & Run All*. A GPU (e.g. Google Colab) is strongly recommended for Task A6.

In [ ]:
import os, time, itertools, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix
warnings.filterwarnings("ignore")

keras.utils.set_random_seed(42)
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

# ---------------- CONFIG ----------------
FAST_MODE = False                     # True -> tiny epochs for a smoke test
EP_SCRATCH = 3 if FAST_MODE else 100  # max epochs (EarlyStopping will cut it)
EP_LENET   = 2 if FAST_MODE else 30
EP_FE      = 1 if FAST_MODE else 20   # feature extraction epochs
EP_FT      = 1 if FAST_MODE else 10   # fine-tuning epochs
BATCH      = 64

CLASS_NAMES = ["Airplane", "Automobile", "Bird", "Cat", "Deer",
               "Dog", "Frog", "Horse", "Ship", "Truck"]
os.makedirs("plots", exist_ok=True)

def save_show(name):
    """Save the current figure to /plots and display it."""
    plt.tight_layout()
    plt.savefig(f"plots/{name}.png", dpi=130, bbox_inches="tight")
    plt.show()

results = []   # collected for the final 6-model comparison table
def add_result(model_name, arch, aug, tl, acc, params):
    results.append({"Model": model_name, "Architecture": arch, "Augmentation": aug,
                    "Transfer Learning": tl, "Test Accuracy": round(float(acc), 4),
                    "Parameters": int(params)})

def count_params(model):
    tr = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    nt = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
    return tr, nt

class LRTracker(keras.callbacks.Callback):
    """Records the learning rate used in every epoch (to mark ReduceLROnPlateau events)."""
    def on_train_begin(self, logs=None):
        self.lrs = []
    def on_epoch_end(self, epoch, logs=None):
        self.lrs.append(float(np.array(self.model.optimizer.learning_rate)))

def get_callbacks():
    es = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True)
    rlrop = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5,
                              min_lr=1e-6, verbose=1)
    lr_tracker = LRTracker()
    return es, rlrop, lr_tracker, [lr_tracker, es, rlrop]

# Task A1 — Data Loading, EDA & Preprocessing

**Objective:** load CIFAR-10, understand its structure, visualise it, normalise pixels and one-hot encode labels.
**Expected output:** printed shapes/dtypes, a 5×10 sample grid, before/after histograms, one-hot labels `(50000, 10)`.

In [ ]:
# A1.1 — Load CIFAR-10
(X_train, y_train), (X_test, y_test) = keras.datasets.cifar10.load_data()
print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_test :", X_test.shape,  "| y_test :", y_test.shape)
print("Unique labels:", np.unique(y_train))
print("dtype:", X_train.dtype)
assert X_train.shape == (50000, 32, 32, 3) and X_test.shape == (10000, 32, 32, 3)
assert X_train.dtype == np.uint8
print("Images per class (train):", np.bincount(y_train.ravel()))

In [ ]:
# A1.2 — 5 x 10 grid: one column per class, 5 rows
fig, axes = plt.subplots(5, 10, figsize=(20, 10.5))
for c in range(10):
    idx = np.where(y_train.ravel() == c)[0][:5]
    for r in range(5):
        ax = axes[r, c]
        ax.imshow(X_train[idx[r]])
        ax.axis("off")
        if r == 0:
            ax.set_title(CLASS_NAMES[c], fontsize=13, fontweight="bold")
fig.suptitle("CIFAR-10 — 5 Samples per Class", fontsize=18, y=1.0)
save_show("a1_sample_grid")

**Observation — visually similar classes.** *Cat* vs *Dog* (similar fur, pose, body shape at 32×32) and *Automobile* vs *Truck* (both boxy vehicles on roads) look alike, and *Deer* vs *Horse* share a similar silhouette. These will be the **hardest pairs** for the CNN, whereas *Airplane* and *Ship* are usually separated by distinctive backgrounds (sky vs water).

In [ ]:
# A1.3 — Pixel distribution before / after normalisation
sample_before = X_train.ravel()[::20]       # strided subsample keeps the plot fast
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(sample_before, bins=64, color="tab:red", alpha=.8)
ax[0].set_title("Pixel values BEFORE normalisation (0-255)")
ax[0].set_xlabel("pixel value"); ax[0].set_ylabel("count")

X_train = X_train.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

ax[1].hist(X_train.ravel()[::20], bins=64, color="tab:green", alpha=.8)
ax[1].set_title("Pixel values AFTER normalisation (0-1)")
ax[1].set_xlabel("pixel value")
save_show("a1_pixel_histograms")
print("min/max after:", X_train.min(), X_train.max(), "| mean:", X_train.mean().round(3))

**WHY normalise?** Gradient descent converges faster when inputs live on [0, 1]. Raw pixel values up to 255 create proportionally large activations and initial weight updates, which **destabilises early training** (exploding pre-activations, saturated non-linearities, a badly conditioned loss surface).

In [ ]:
# A1.4 — One-hot encode labels
y_train_ohe = keras.utils.to_categorical(y_train, 10)
y_test_ohe  = keras.utils.to_categorical(y_test, 10)
print("y_train_ohe:", y_train_ohe.shape, "| y_test_ohe:", y_test_ohe.shape)
print("Label", y_train[0][0], "->", y_train_ohe[0])

# Fixed train / validation split (last 10 %) used for EVERY model so results are comparable
N_TR = int(len(X_train) * 0.9)
X_tr,  y_tr  = X_train[:N_TR],  y_train_ohe[:N_TR]
X_val, y_val = X_train[N_TR:],  y_train_ohe[N_TR:]
print("train:", X_tr.shape, "| val:", X_val.shape)

**WHY one-hot for the CNN output?** The softmax output layer produces a probability distribution over the 10 classes; `categorical_crossentropy` compares that distribution with the one-hot ground truth. Integer labels would instead require `sparse_categorical_crossentropy`.

### CNN vs ANN for Image Classification

1. **ANN (MLP)** flattens the image to a 1-D vector — it **loses all spatial relationships** between neighbouring pixels.
2. **CNN** slides a small learnable filter (kernel) over the 2-D image; each filter detects a local pattern **wherever it appears** → *translation invariance*.
3. **Parameter sharing:** the same filter weights are reused at every spatial position, so a CNN has far fewer parameters than an equivalent MLP.

**Example.** A 3×3 filter on a 32×32 image shares the *same 9 weights* across all 30×30 = 900 valid positions, whereas an MLP covering the same positions would need an independent weight per connection. A single dense layer on the raw image (3072 inputs → 256 units) alone has ~786 K weights; a whole conv layer with 32 filters of 3×3×3 has only **896** parameters.

In [ ]:
# Quantify the parameter-sharing argument
mlp_first_layer = 32*32*3*256 + 256
conv_layer      = 3*3*3*32 + 32
print(f"MLP first Dense(256) on flattened image : {mlp_first_layer:,} params")
print(f"Conv2D(32, 3x3) on 32x32x3 image        : {conv_layer:,} params")
print(f"Ratio                                   : {mlp_first_layer/conv_layer:,.0f}x")

# Task A2 — Convolution Operation, Padding & Strides

**Objective:** understand convolution by hand first, then see how padding and strides change shapes, and finally inspect the feature maps a Conv2D layer produces.
**Expected output:** original vs edge-filtered image, shape tables for VALID/SAME and strides, 4×4 grid of 16 feature maps.

In [ ]:
# A2.1 — Manual convolution with SciPy on a greyscale CIFAR image
from scipy.signal import convolve2d

img = X_train[0]
gray = np.mean(img, axis=2)
kernel = np.array([[-1, -1, -1],
                   [-1,  8, -1],
                   [-1, -1, -1]], dtype="float32")
edges = convolve2d(gray, kernel, mode="valid")

fig, ax = plt.subplots(1, 3, figsize=(14, 4.5))
ax[0].imshow(img);                 ax[0].set_title(f"Original RGB ({CLASS_NAMES[int(y_train[0][0])]})")
ax[1].imshow(gray, cmap="gray");   ax[1].set_title(f"Greyscale {gray.shape}")
ax[2].imshow(edges, cmap="gray");  ax[2].set_title(f"Edge-detection output {edges.shape}")
for a in ax: a.axis("off")
save_show("a2_manual_convolution")

# Verify one output pixel by hand: dot product of kernel with the flipped patch
patch = gray[0:3, 0:3]
manual = np.sum(patch * np.flip(kernel))        # true convolution flips the kernel
print("Manual dot product at (0,0):", round(float(manual), 5), "| scipy:", round(float(edges[0, 0]), 5))

**Convolution = dot product of the kernel with every overlapping image patch.** The kernel slides one step at a time (the *stride*), producing a new *feature map*. This exact operation is what `Conv2D` layers perform on **every input channel simultaneously**, except that the kernel values are **learned** by backpropagation instead of being hand-designed. (Strictly, deep-learning libraries compute *cross-correlation*, i.e. no kernel flip — irrelevant because the kernels are learned.)

In [ ]:
# A2.2 — Padding: VALID vs SAME
def conv_out_shape(padding, strides=1, kernel=3):
    m = keras.Sequential([keras.Input((32, 32, 3)),
                          layers.Conv2D(8, kernel, strides=strides, padding=padding)])
    return m.output_shape

print("Formula (VALID): out = floor((in - k) / s) + 1   ;   (SAME): out = ceil(in / s)\n")
print("VALID, k=3, s=1 ->", conv_out_shape("valid"), "  formula:", (32 - 3) // 1 + 1)
print("SAME , k=3, s=1 ->", conv_out_shape("same"),  "  formula:", int(np.ceil(32 / 1)))

fig, ax = plt.subplots(1, 2, figsize=(10, 4.5))
ax[0].imshow(img);                                              ax[0].set_title("Original 32x32")
ax[1].imshow(np.pad(img, ((1, 1), (1, 1), (0, 0))));            ax[1].set_title("Zero-padded (SAME, pad=1) -> 34x34")
for a in ax: a.axis("off")
save_show("a2_padding")

**SAME padding** zero-pads the border so the output has the same spatial size as the input. Most CNN architectures use it so that **many conv layers can be stacked without the feature maps shrinking too quickly** (VALID loses `k-1` pixels per layer).

In [ ]:
# A2.3 — Strides
print(f"{'padding':8s} {'stride':6s} {'output shape':18s} formula")
for padding in ["valid", "same"]:
    for s in [1, 2]:
        f = (32 - 3) // s + 1 if padding == "valid" else int(np.ceil(32 / s))
        print(f"{padding:8s} {s:<6d} {str(conv_out_shape(padding, s)):18s} {f}x{f}")

**Stride** is the number of pixels the filter moves at each step. `stride=1` gives dense feature maps; `stride=2` **halves** the spatial size (SAME: 32→16) — a *learned* alternative to pooling, instead of a fixed max/mean rule. Modern detectors such as YOLOv8 use strided convolutions rather than max-pooling for downsampling.

In [ ]:
# A2.4 — Build a single Conv2D layer and visualise its 16 feature maps
model_vis = keras.Sequential([
    keras.Input((32, 32, 3)),
    layers.Conv2D(16, (3, 3), activation="relu", padding="same", name="conv_vis")
])
x_one = X_train[:1]                               # one image, shape (1,32,32,3)
fmaps = model_vis.predict(x_one, verbose=0)
print("Feature-map tensor:", fmaps.shape)         # (1, 32, 32, 16)

def show_maps(fm, title, name, rows=4, cols=4):
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.2, rows * 2.3))
    for i, ax in enumerate(axes.ravel()):
        ax.imshow(fm[0, :, :, i], cmap="viridis"); ax.axis("off"); ax.set_title(f"filter {i}", fontsize=9)
    fig.suptitle(title, fontsize=14)
    save_show(name)

show_maps(fmaps, "16 Feature Maps from a Single Conv2D(16, 3x3, SAME) Layer", "a2_feature_maps")

Each feature map shows what **one learnable filter responds to**. Early filters detect edges, gradients and colours; deeper layers (in a trained network) detect progressively more complex patterns — textures, shapes and object parts. These filters are still *random* (untrained), so they act like random edge/colour detectors — the training in Task A4 makes them meaningful.

# Task A3 — Pooling, MaxPooling, Flatten & CNN Architecture

**Objective:** compare pooling types, understand Flatten, then build LeNet-5 (1998) and a modern CNN.
**Expected output:** before/after pooled maps (32×32 → 16×16), Flatten shapes, two model summaries with verified parameter counts.

In [ ]:
# A3.1 — MaxPooling vs AveragePooling
fm = tf.constant(fmaps)
max_pooled = layers.MaxPooling2D((2, 2))(fm).numpy()
avg_pooled = layers.AveragePooling2D((2, 2))(fm).numpy()
print("Before pooling :", fmaps.shape)
print("After MaxPool  :", max_pooled.shape)
print("After AvgPool  :", avg_pooled.shape)

show_maps(fmaps,      "Feature maps BEFORE pooling (32x32)",       "a3_before_pooling")
show_maps(max_pooled, "AFTER MaxPooling2D(2,2) (16x16)",           "a3_after_maxpool")
show_maps(avg_pooled, "AFTER AveragePooling2D(2,2) (16x16)",       "a3_after_avgpool")

1. **MaxPooling2D(2,2)** keeps the maximum of each 2×2 patch → retains the **strongest activation** (most prominent feature) and gives slight translation invariance; outputs look *sharper*.
2. **AveragePooling2D(2,2)** takes the mean → *smoother*, preserves more background/spatial information, but dilutes strong features.
3. Modern CNNs often replace Flatten with **GlobalAveragePooling2D**, which averages each feature map to a single number — dramatically fewer parameters in the dense head.

In [ ]:
# A3.2 — Flatten
vis_chain = keras.Sequential([
    keras.Input((32, 32, 3)),
    layers.Conv2D(16, 3, activation="relu", padding="same"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
])
for l in vis_chain.layers:
    print(f"{l.name:20s} output -> {l.output.shape}")
print("\nFlatten: (None,16,16,16) -> (None,4096) because 16*16*16 =", 16*16*16)
print("Dense(256) after this Flatten would need", 4096*256 + 256, "parameters")
print("Dense(256) after GlobalAveragePooling2D (16 values) needs", 16*256 + 256)

**Flatten** converts the 3-D tensor (H, W, C) into a 1-D vector so it can be connected to a Dense classification head. The flattened size fixes the number of parameters in the first Dense layer — which is why **spatial dimensions must be reduced (pooling) before Flatten** to keep parameter counts manageable (compare 4096-input vs 16-input Dense above).

In [ ]:
# A3.3 — LeNet-5 (LeCun et al., 1998) adapted to CIFAR-10
def build_lenet5():
    return keras.Sequential([
        keras.Input((32, 32, 3)),
        layers.Conv2D(6, (5, 5), activation="tanh", name="C1"),
        layers.AveragePooling2D((2, 2), name="S2"),
        layers.Conv2D(16, (5, 5), activation="tanh", name="C3"),
        layers.AveragePooling2D((2, 2), name="S4"),
        layers.Flatten(name="flatten"),
        layers.Dense(120, activation="tanh", name="F5"),
        layers.Dense(84, activation="tanh", name="F6"),
        layers.Dense(10, activation="softmax", name="output"),
    ], name="LeNet5")

lenet = build_lenet5()
lenet.summary()

# Manual parameter counts: (kh*kw*in_channels + 1 bias) * filters
c1 = (5*5*3 + 1) * 6
c3 = (5*5*6 + 1) * 16
f5 = (5*5*16) * 120 + 120          # flatten size: 32 -conv5-> 28 -pool-> 14 -conv5-> 10 -pool-> 5  => 5*5*16 = 400
f6 = 120 * 84 + 84
out = 84 * 10 + 10
print(f"C1 = (5*5*3+1)*6      = {c1}")
print(f"C3 = (5*5*6+1)*16     = {c3}")
print(f"F5 = 400*120+120      = {f5}")
print(f"F6 = 120*84+84        = {f6}")
print(f"Output = 84*10+10     = {out}")
print("Manual total:", c1 + c3 + f5 + f6 + out, "| Keras total:", lenet.count_params())
assert c1 + c3 + f5 + f6 + out == lenet.count_params()
assert lenet.get_layer("C1").count_params() == c1 and lenet.get_layer("C3").count_params() == c3

**LeNet-5 (1998)** was the first successful CNN, used to read handwritten US postal codes and bank cheques. It established the pattern **Conv → Pool → Conv → Pool → Flatten → Dense → Dense** that is still used today. Differences from modern CNNs: **tanh** instead of ReLU, **AveragePooling** instead of MaxPooling, large **5×5 filters** (modern nets stack 3×3 filters), no batch-norm/dropout, and tiny capacity (~62 K parameters).

In [ ]:
# Train LeNet-5 (needed for the final comparison table)
lenet.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
es_l = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
t0 = time.time()
hist_lenet = lenet.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                       epochs=EP_LENET, batch_size=128, callbacks=[es_l], verbose=2)
_, acc_lenet = lenet.evaluate(X_test, y_test_ohe, verbose=0)
print(f"LeNet-5 test accuracy: {acc_lenet:.4f}  ({time.time()-t0:.0f}s)")
add_result("LeNet-5", "Conv-AvgPool x2 + Dense (tanh)", "No", "No", acc_lenet, lenet.count_params())

In [ ]:
# A3.4 — Modern custom CNN baseline (VGG-style: 2 convs per block)
def build_cnn(aug=None, name="CNN_scratch"):
    inp = keras.Input((32, 32, 3))
    x = aug(inp) if aug is not None else inp
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(32, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    out = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inp, out, name=name)

model_cnn = build_cnn()
model_cnn.summary()
assert model_cnn.count_params() < 2_000_000
print("Parameters:", f"{model_cnn.count_params():,}", "< 2,000,000  OK")

**WHY stack two conv layers before each pooling layer?** The first conv detects simple features (edges, colour blobs); the second **composes them into slightly more complex patterns** before the spatial resolution is reduced. Two stacked 3×3 convs have the same receptive field as one 5×5 conv but with fewer parameters (2·9·C² vs 25·C²) and an extra non-linearity — a richer feature hierarchy at each scale.

# Task A4 — Backpropagation in CNN & Training

**Objective:** explain gradient flow in a CNN, train the baseline with callbacks, and analyse its errors.
**Expected output:** annotated training curves, test accuracy, 10×10 confusion matrix, top-3 confused pairs, 10 misclassified images.

### Backpropagation in a CNN

1. **Forward pass:** input image → convolve with learnable kernels → activation → pool → (repeat) → flatten → dense → softmax → loss.
2. **Backward pass** — gradients flow backwards via the chain rule:
   - ∂Loss/∂W for **Dense** layers is identical to ordinary MLP backprop.
   - ∂Loss/∂W for **Conv** layers: the gradient w.r.t. each element of each kernel is obtained by **correlating the upstream gradient with the local input patch** (summed over all spatial positions, because the kernel is shared). The gradient w.r.t. the layer input is the upstream gradient convolved with the 180°-rotated kernel.
3. **Gradient through MaxPooling:** only the position that produced the maximum in the forward pass receives the full gradient; all other positions get **zero** — the *max-location switch* (max-index mask).
4. **Gradient through AveragePooling:** the upstream gradient is **divided equally** among all positions of the pooling window (each gets 1/4 for a 2×2 window).
5. **Gradient through Flatten** is trivial: it simply **reshapes** the 1-D gradient back to 3-D.

In [ ]:
# Demonstration: gradient through MaxPooling is a one-hot "switch"; through AvgPool it is uniform
patch = tf.Variable([[[[1.0], [5.0]], [[3.0], [2.0]]]])       # shape (1,2,2,1)
with tf.GradientTape(persistent=True) as tape:
    y_max = layers.MaxPooling2D(2)(patch)
    y_avg = layers.AveragePooling2D(2)(patch)
print("Input 2x2 patch:\n", patch.numpy()[0, :, :, 0])
print("dMax/dInput  (only the max position gets gradient):\n", tape.gradient(y_max, patch).numpy()[0, :, :, 0])
print("dAvg/dInput  (gradient split equally):\n", tape.gradient(y_avg, patch).numpy()[0, :, :, 0])

In [ ]:
# A4.2 — Compile and train the CNN (no augmentation)
model_cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
es, rlrop, lr_tr, cbs = get_callbacks()
t0 = time.time()
history = model_cnn.fit(X_tr, y_tr, validation_data=(X_val, y_val),
                        epochs=EP_SCRATCH, batch_size=BATCH, callbacks=cbs, verbose=2)
print(f"Training time: {(time.time()-t0)/60:.1f} min | epochs run: {len(history.history['loss'])}")

`ReduceLROnPlateau` **halves the learning rate when `val_loss` stops improving**, letting the model move from a coarse search (high lr) to fine-tuning (low lr) without manual intervention. `EarlyStopping(restore_best_weights=True)` stops training once validation loss has not improved for 10 epochs and rolls back to the best weights.

In [ ]:
# A4.3 — Training curves with EarlyStopping + ReduceLROnPlateau markers
def plot_curves(hist, es_cb, lr_cb, title, fname):
    h = hist.history
    ep = np.arange(1, len(h["loss"]) + 1)
    stop_ep = es_cb.stopped_epoch + 1 if es_cb.stopped_epoch > 0 else None
    lr_drops = [i + 1 for i in range(1, len(lr_cb.lrs)) if lr_cb.lrs[i] < lr_cb.lrs[i - 1] - 1e-12]

    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    for a, key, label in zip(ax, ["loss", "accuracy"], ["Loss", "Accuracy"]):
        a.plot(ep, h[key], label=f"train {key}", lw=2)
        a.plot(ep, h["val_" + key], label=f"val {key}", lw=2)
        for k, e in enumerate(lr_drops):
            a.axvline(e, color="grey", ls="--", alpha=.7, label="LR reduced" if k == 0 else None)
        if stop_ep:
            a.axvline(stop_ep, color="red", ls="--", lw=2, label=f"EarlyStopping @ epoch {stop_ep}")
        a.set_xlabel("epoch"); a.set_ylabel(label); a.grid(alpha=.3); a.legend()
        a.annotate(f"train {h[key][-1]:.3f}", (ep[-1], h[key][-1]), textcoords="offset points",
                   xytext=(-45, 10), fontsize=10, color="tab:blue")
        a.annotate(f"val {h['val_' + key][-1]:.3f}", (ep[-1], h["val_" + key][-1]), textcoords="offset points",
                   xytext=(-45, -18), fontsize=10, color="tab:orange")
    ax[0].set_title("Loss"); ax[1].set_title("Accuracy")
    fig.suptitle(title, fontsize=15)
    save_show(fname)
    print("Stopped (EarlyStopping) at epoch:", stop_ep, "| LR reduced at epochs:", lr_drops)

plot_curves(history, es, lr_tr, "CNN Training Curves — CIFAR-10", "a4_training_curves")

In [ ]:
# A4.4 — Evaluate and analyse
test_loss, test_acc = model_cnn.evaluate(X_test, y_test_ohe, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")
add_result("CNN Scratch (No Aug)", "VGG-style 5 conv + GAP", "No", "No", test_acc, model_cnn.count_params())

probs = model_cnn.predict(X_test, verbose=0)
y_pred = probs.argmax(axis=1)
y_true = y_test.ravel()

def plot_confusion(y_true, y_pred, title, fname):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(10)); ax.set_xticklabels(CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(10)); ax.set_yticklabels(CLASS_NAMES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(title)
    for i, j in itertools.product(range(10), range(10)):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=9,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
    fig.colorbar(im, fraction=.046)
    save_show(fname)
    return cm

cm = plot_confusion(y_true, y_pred, f"Confusion Matrix — CNN (test acc {test_acc:.3f})", "a4_confusion_matrix")

# Top-3 most confused (unordered) class pairs
pairs = []
for i in range(10):
    for j in range(i + 1, 10):
        pairs.append((cm[i, j] + cm[j, i], CLASS_NAMES[i], CLASS_NAMES[j]))
pairs.sort(reverse=True)
print("Top-3 most confused class pairs:")
for n, a, b in pairs[:3]:
    print(f"  {a:10s} <-> {b:10s} : {n} errors")

# Per-class accuracy
per_class = cm.diagonal() / cm.sum(axis=1)
print("\nPer-class accuracy:")
for c, a in sorted(zip(CLASS_NAMES, per_class), key=lambda t: t[1]):
    print(f"  {c:10s} {a:.3f}")

In [ ]:
# 10 misclassified images (2 x 5)
wrong = np.where(y_pred != y_true)[0]
rng = np.random.default_rng(42)
pick = rng.choice(wrong, 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(14, 6.5))
for ax, i in zip(axes.ravel(), pick):
    ax.imshow(X_test[i]); ax.axis("off")
    ax.set_title(f"True: {CLASS_NAMES[y_true[i]]}\nPred: {CLASS_NAMES[y_pred[i]]} ({probs[i].max():.2f})",
                 fontsize=10, color="crimson")
fig.suptitle("Misclassified Test Images", fontsize=15)
save_show("a4_misclassified")

**Why are cat/dog and automobile/truck confused more than airplane/ship?** At 32×32 resolution a lot of fine detail is lost: ears, snouts, fur texture and wheel/cab details that separate cats from dogs or cars from trucks are only a few pixels wide, while both classes share the same colours, backgrounds and overall body layout. Airplanes (sky) and ships (water) are usually separable by **global context and colour distribution**, so even a small CNN handles them well.

# Task A5 — Data Augmentation

**Objective:** reduce overfitting by artificially expanding the training set.
**Expected output:** augmentation preview (original + 9 variants), 4-line comparison plot (no-aug vs aug), Keras-layer augmentation result.

### WHY data augmentation?
A CNN with ~0.2–2 M parameters trained on 50 000 images risks **memorising** the training set. Augmentation applies random transformations (flip, shift, rotation, zoom) to each image **at training time**, so the model sees a *different* version of every image each epoch. This acts as a strong **regulariser**, reducing overfitting without collecting more data. It is especially important for small datasets (< 100 K images) and is standard practice in production computer vision.

In [ ]:
# A5.2 — ImageDataGenerator augmentation + visualisation
try:
    from tensorflow.keras.preprocessing.image import ImageDataGenerator
except Exception:
    from keras.src.legacy.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(rotation_range=15, width_shift_range=0.1, height_shift_range=0.1,
                             horizontal_flip=True, zoom_range=0.1, fill_mode="nearest")
datagen.fit(X_tr)

sample = X_train[7]
flow = datagen.flow(sample[np.newaxis, ...], batch_size=1, seed=1)
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
axes[0, 0].imshow(sample); axes[0, 0].set_title("ORIGINAL", fontweight="bold")
axes[0, 0].axis("off")
for ax in axes.ravel()[1:]:
    ax.imshow(np.clip(next(flow)[0], 0, 1)); ax.axis("off"); ax.set_title("augmented", fontsize=10)
fig.suptitle("Data Augmentation — 9 Random Transformations of One CIFAR-10 Image", fontsize=15)
save_show("a5_augmentation_grid")

In [ ]:
# A5.3 — Retrain the SAME architecture with ImageDataGenerator
model_aug = build_cnn(name="CNN_aug_IDG")
model_aug.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
es_a, rlrop_a, lr_a, cbs_a = get_callbacks()
train_flow = datagen.flow(X_tr, y_tr, batch_size=BATCH, seed=42)
t0 = time.time()
history_aug = model_aug.fit(train_flow, steps_per_epoch=len(X_tr) // BATCH,
                            validation_data=(X_val, y_val), epochs=EP_SCRATCH,
                            callbacks=cbs_a, verbose=2)
print(f"Training time: {(time.time()-t0)/60:.1f} min | epochs: {len(history_aug.history['loss'])}")
_, acc_aug = model_aug.evaluate(X_test, y_test_ohe, verbose=0)
print(f"Test accuracy WITH augmentation: {acc_aug:.4f}  (without: {test_acc:.4f})")
add_result("CNN Scratch + Aug (IDG)", "VGG-style 5 conv + GAP", "ImageDataGenerator", "No", acc_aug, model_aug.count_params())

In [ ]:
# Comparison plot: no-aug vs aug (4 lines per subplot pair -> loss: 4 lines, accuracy: 4 lines)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for a, key, title in zip(ax, ["loss", "accuracy"], ["Loss", "Accuracy"]):
    a.plot(history.history[key],            label="no-aug train",  color="tab:blue",   ls="-")
    a.plot(history.history["val_" + key],   label="no-aug val",    color="tab:blue",   ls="--")
    a.plot(history_aug.history[key],        label="aug train",     color="tab:orange", ls="-")
    a.plot(history_aug.history["val_" + key], label="aug val",     color="tab:orange", ls="--")
    a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend()
fig.suptitle("No Augmentation vs ImageDataGenerator Augmentation", fontsize=15)
save_show("a5_comparison_aug_vs_noaug")

gap_no  = history.history["accuracy"][-1] - history.history["val_accuracy"][-1]
gap_aug = history_aug.history["accuracy"][-1] - history_aug.history["val_accuracy"][-1]
print(f"Train-val accuracy gap: no-aug = {gap_no:+.3f} | aug = {gap_aug:+.3f}")

The augmented model should show a **smaller gap between training and validation accuracy** — i.e. less overfitting. Training accuracy may be slightly *lower* (each batch is a harder, transformed version of the data) but **test accuracy should be higher**, because the model learned more general features.

In [ ]:
# A5.4 — Modern augmentation with Keras preprocessing layers (active only during training)
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.1, 0.1),
], name="data_augmentation")

model_aug2 = build_cnn(aug=data_augmentation, name="CNN_aug_layers")
model_aug2.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
es_b, rlrop_b, lr_b, cbs_b = get_callbacks()
t0 = time.time()
history_aug2 = model_aug2.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=EP_SCRATCH,
                              batch_size=BATCH, callbacks=cbs_b, verbose=2)
print(f"Training time: {(time.time()-t0)/60:.1f} min | epochs: {len(history_aug2.history['loss'])}")
_, acc_aug2 = model_aug2.evaluate(X_test, y_test_ohe, verbose=0)
print(f"Test accuracy: ImageDataGenerator = {acc_aug:.4f} | Keras layers = {acc_aug2:.4f}")
add_result("CNN + Keras Aug Layers", "VGG-style 5 conv + GAP", "Keras preprocessing layers", "No", acc_aug2, model_aug2.count_params())
plot_curves(history_aug2, es_b, lr_b, "CNN + Keras Augmentation Layers — Training Curves", "a5_keras_aug_curves")

Keras preprocessing layers live **inside the model**: they are active during training and automatically **inactive at inference**, no separate `datagen.flow()` needed. Because they run on the GPU as part of the graph they are also faster and make the exported model self-contained.

# Task A6 — Transfer Learning: Feature Extraction & Fine-Tuning

**Objective:** reuse an ImageNet-pre-trained MobileNetV2 for CIFAR-10, compare it to the scratch CNNs, and demonstrate a residual connection.
**Expected output:** trainable vs frozen parameter counts, FE + FT accuracy, concatenated history plot, full 6-model table, residual block.

### WHY Transfer Learning?
Training a deep CNN from scratch needs lots of data and compute. **Transfer learning** adapts a model pre-trained on a far larger dataset (ImageNet: ~14 M images, 1 000 classes). Its convolutional stack already encodes a hierarchy of visual features — edges → textures → shapes → object parts — that is useful across vision tasks.

1. **Feature Extraction** — freeze all pre-trained weights, add a new classification head, train *only the head*. Fast (minutes) but less accurate.
2. **Fine-Tuning** — unfreeze the top *N* layers of the base and train them with the head at a **very low learning rate**. Slower, but noticeably more accurate.

In [ ]:
# A6.2 — Feature Extraction with MobileNetV2
# CIFAR-10 is 32x32; MobileNetV2 needs >= 96x96. Resizing the full set at once would take ~5.5 GB of RAM,
# so images are resized ON THE FLY inside a tf.data pipeline (same result as tf.image.resize, much lower memory).
IMG = 96
def make_ds(X, y, training=False, bs=64):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if training:
        ds = ds.shuffle(10_000, seed=42)
    ds = ds.map(lambda a, b: (tf.image.resize(a, (IMG, IMG)) * 2.0 - 1.0, b),   # [0,1] -> [-1,1] (MobileNetV2 range)
                num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(bs).prefetch(tf.data.AUTOTUNE)

train_ds = make_ds(X_tr,  y_tr,  training=True, bs=64)
val_ds   = make_ds(X_val, y_val, bs=64)
test_ds  = make_ds(X_test, y_test_ohe, bs=64)

base_model = keras.applications.MobileNetV2(input_shape=(IMG, IMG, 3), include_top=False, weights="imagenet")
base_model.trainable = False

inputs = keras.Input(shape=(IMG, IMG, 3))
x = base_model(inputs, training=False)            # keep BatchNorm in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
outputs = layers.Dense(10, activation="softmax")(x)
model_fe = keras.Model(inputs, outputs, name="MobileNetV2_FE")

tr, nt = count_params(model_fe)
print(f"Total params     : {tr + nt:,}")
print(f"Trainable params : {tr:,}   ({100*tr/(tr+nt):.1f} %)")
print(f"Frozen params    : {nt:,}   ({100*nt/(tr+nt):.1f} %)")

In [ ]:
# A6.3 — Train the feature-extraction model
model_fe.compile(optimizer=keras.optimizers.Adam(0.001), loss="categorical_crossentropy", metrics=["accuracy"])
t0 = time.time()
history_fe = model_fe.fit(train_ds, validation_data=val_ds, epochs=EP_FE, verbose=2)
print(f"Training time: {(time.time()-t0)/60:.1f} min")
_, acc_fe = model_fe.evaluate(test_ds, verbose=0)
print(f"MobileNetV2 Feature Extraction test accuracy: {acc_fe:.4f}")
add_result("MobileNetV2 Feature Extraction", "MobileNetV2 (frozen) + GAP + Dense", "No", "Feature Extraction",
           acc_fe, model_fe.count_params())

In [ ]:
# A6.4 — Fine-tuning: unfreeze the top 30 layers, very low learning rate
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

tr2, nt2 = count_params(model_fe)
print(f"Trainable params after unfreezing top 30 layers: {tr2:,} (was {tr:,})")

model_fe.compile(optimizer=keras.optimizers.Adam(1e-5), loss="categorical_crossentropy", metrics=["accuracy"])
train_ds32 = make_ds(X_tr, y_tr, training=True, bs=32)
t0 = time.time()
history_ft = model_fe.fit(train_ds32, validation_data=val_ds, epochs=EP_FT, verbose=2)
print(f"Fine-tuning time: {(time.time()-t0)/60:.1f} min")
_, acc_ft = model_fe.evaluate(test_ds, verbose=0)
print(f"MobileNetV2 Fine-Tuned test accuracy: {acc_ft:.4f}  (feature extraction was {acc_fe:.4f})")
add_result("MobileNetV2 Fine-Tuned", "MobileNetV2 (top-30 unfrozen) + GAP + Dense", "No", "Fine-Tuning",
           acc_ft, model_fe.count_params())

In [ ]:
# Concatenated history: feature extraction + fine-tuning
acc_all  = history_fe.history["accuracy"]     + history_ft.history["accuracy"]
vacc_all = history_fe.history["val_accuracy"] + history_ft.history["val_accuracy"]
loss_all = history_fe.history["loss"]         + history_ft.history["loss"]
vloss_all = history_fe.history["val_loss"]    + history_ft.history["val_loss"]
split = len(history_fe.history["loss"])
ep = np.arange(1, len(acc_all) + 1)

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
for a, tr_, va_, title in zip(ax, [loss_all, acc_all], [vloss_all, vacc_all], ["Loss", "Accuracy"]):
    a.plot(ep, tr_, label="train", lw=2); a.plot(ep, va_, label="val", lw=2)
    a.axvline(split + 0.5, color="red", ls="--", lw=2, label="Fine-tuning starts")
    a.set_title(title); a.set_xlabel("epoch"); a.grid(alpha=.3); a.legend()
fig.suptitle("MobileNetV2 — Feature Extraction → Fine-Tuning", fontsize=15)
save_show("a6_finetuning_history")

**WHY a very low learning rate (1e-5) for fine-tuning?** The pre-trained weights are already good. A large learning rate would produce big updates that **destroy** the learned features (catastrophic forgetting) — especially because the new head was only just trained and still sends noisy gradients into the base. Fine-tuning should only **nudge** the top layers towards the new task. The base is called with `training=False`, so BatchNorm layers keep their ImageNet statistics.

In [ ]:
# A6.5 — Full results comparison table
df = pd.DataFrame(results)
order = ["LeNet-5", "CNN Scratch (No Aug)", "CNN Scratch + Aug (IDG)", "CNN + Keras Aug Layers",
         "MobileNetV2 Feature Extraction", "MobileNetV2 Fine-Tuned"]
df["Model"] = pd.Categorical(df["Model"], categories=order, ordered=True)
df = df.sort_values("Model").reset_index(drop=True)
df["Model"] = df["Model"].astype(str)
df.to_csv("plots/results_table.csv", index=False)

# Save the table as an image for the README
fig, ax = plt.subplots(figsize=(15, 3))
ax.axis("off")
tbl = ax.table(cellText=df.values, colLabels=df.columns, loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(9); tbl.scale(1, 1.8)
best_row = int(df["Test Accuracy"].idxmax()) + 1
for c in range(len(df.columns)):
    tbl[(best_row, c)].set_facecolor("#b7e4c7")
    tbl[(0, c)].set_facecolor("#d9d9d9")
save_show("a6_results_table")

df.style.highlight_max(subset=["Test Accuracy"], color="lightgreen").format({"Parameters": "{:,}", "Test Accuracy": "{:.4f}"})

In [ ]:
# Accuracy vs parameters (effort / accuracy trade-off)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
colors = plt.cm.tab10(np.arange(len(df)))
ax[0].barh(df["Model"], df["Test Accuracy"], color=colors)
ax[0].set_xlim(max(0, df["Test Accuracy"].min() - .05), 1); ax[0].invert_yaxis()
for i, v in enumerate(df["Test Accuracy"]):
    ax[0].text(v + .002, i, f"{v:.3f}", va="center")
ax[0].set_title("Test accuracy per model"); ax[0].set_xlabel("accuracy")
ax[1].scatter(df["Parameters"], df["Test Accuracy"], s=150, c=colors)
for _, r in df.iterrows():
    ax[1].annotate(r["Model"], (r["Parameters"], r["Test Accuracy"]), fontsize=8, xytext=(5, 5), textcoords="offset points")
ax[1].set_xscale("log"); ax[1].set_xlabel("parameters (log scale)"); ax[1].set_ylabel("test accuracy")
ax[1].set_title("Accuracy vs model size"); ax[1].grid(alpha=.3)
save_show("a6_accuracy_summary")

**Accuracy progression & effort/accuracy trade-off**

| Approach | Effort | Typical outcome |
|---|---|---|
| LeNet-5 | seconds–minutes, tiny model | modest accuracy — limited capacity, tanh + 5×5 filters |
| CNN from scratch | tens of minutes on CPU | big jump from LeNet: ReLU + stacked 3×3 + depth |
| + Augmentation (IDG / Keras layers) | same model, longer to converge | smaller train-val gap and higher test accuracy |
| MobileNetV2 Feature Extraction | few minutes, only the head is trained | already competitive with / better than scratch CNNs thanks to ImageNet features |
| MobileNetV2 Fine-Tuning | extra minutes, low lr | the best accuracy, at the cost of more compute and care (lr, which layers to unfreeze) |

*(Exact numbers are in the table above — they depend on your run.)*

In [ ]:
# Confusion matrix of the best transfer-learning model + save best model as .h5
best_name = df.loc[df["Test Accuracy"].idxmax(), "Model"]
print("Best model:", best_name)

probs_ft = model_fe.predict(test_ds, verbose=0)
cm_ft = plot_confusion(y_true, probs_ft.argmax(axis=1), f"Confusion Matrix — MobileNetV2 Fine-Tuned (acc {acc_ft:.3f})",
                       "a6_confusion_matrix_finetuned")

model_fe.save("cifar10_mobilenetv2_finetuned.h5")
model_aug2.save("cifar10_cnn_scratch_aug.h5")
print("Saved: cifar10_mobilenetv2_finetuned.h5, cifar10_cnn_scratch_aug.h5")

In [ ]:
# A6.6 — Non-linear architecture: a minimal residual block (Functional API)
def residual_block(x, filters=64):
    shortcut = layers.Conv2D(filters, 1, padding="same", name="shortcut_1x1")(x)   # match channels
    y = layers.Conv2D(filters, 3, padding="same", activation="relu", name="res_conv1")(x)
    y = layers.Conv2D(filters, 3, padding="same", activation="relu", name="res_conv2")(y)
    y = layers.Add(name="skip_add")([shortcut, y])
    return layers.Activation("relu", name="res_out")(y)

inp = keras.Input((32, 32, 3))
r = residual_block(inp, 64)
r = layers.GlobalAveragePooling2D()(r)
out = layers.Dense(10, activation="softmax")(r)
res_model = keras.Model(inp, out, name="Residual_Block_Demo")
res_model.summary()

# Show the non-sequential topology: Add has two inputs
add_layer = res_model.get_layer("skip_add")
print("\nAdd layer receives", len(add_layer.input), "inputs:", [t.shape for t in add_layer.input])

**Non-linear (non-sequential) architectures** allow *skip connections*: because `output = F(x) + shortcut(x)`, the gradient has a direct additive path (∂/∂x = ∂F/∂x + 1 for an identity shortcut) that **bypasses problematic layers** during backprop. This counters the **vanishing-gradient** problem and makes very deep networks (ResNet, He et al., 2016 — 50, 101, 152 layers) trainable. Here the shortcut uses a 1×1 convolution to match the channel count before the `Add`.

In [ ]:
# Bonus: tiny head-to-head — does the residual connection help trainability? (short run)
def build_mini(residual=True):
    i = keras.Input((32, 32, 3))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(i)
    for f in [32, 64]:
        if residual:
            s = layers.Conv2D(f, 1, padding="same")(x)
            y = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
            y = layers.Conv2D(f, 3, padding="same")(y)
            x = layers.Activation("relu")(layers.Add()([s, y]))
        else:
            x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
            x = layers.Conv2D(f, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    return keras.Model(i, layers.Dense(10, activation="softmax")(x))

bonus = {}
for flag in [False, True]:
    m = build_mini(flag)
    m.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
    h = m.fit(X_tr, y_tr, validation_data=(X_val, y_val), epochs=1 if FAST_MODE else 8, batch_size=128, verbose=0)
    bonus[flag] = h.history["val_accuracy"]
    print(f"{'Residual' if flag else 'Plain   '} mini-net  final val acc: {bonus[flag][-1]:.4f}")

plt.figure(figsize=(7, 4.5))
plt.plot(bonus[False], marker="o", label="plain"); plt.plot(bonus[True], marker="o", label="residual")
plt.xlabel("epoch"); plt.ylabel("val accuracy"); plt.title("Plain vs Residual mini-network"); plt.grid(alpha=.3); plt.legend()
save_show("a6_residual_vs_plain")

# Conclusions

* **Spatial structure matters:** convolution + weight sharing gives a CNN far fewer parameters than an MLP and translation invariance.
* **Architecture:** LeNet-5 → stacked 3×3 ReLU conv blocks + MaxPooling + GlobalAveragePooling is a large improvement.
* **Regularisation:** augmentation shrinks the train/val gap; callbacks (EarlyStopping, ReduceLROnPlateau) automate training length and learning-rate scheduling.
* **Transfer learning** gives the best accuracy per unit of effort; **fine-tuning** at a tiny learning rate squeezes out extra accuracy.
* **Residual connections** provide gradient shortcuts that make deeper networks trainable.
* **Hardest classes:** cat/dog and automobile/truck — a limitation of 32×32 resolution.

**Files produced:** `plots/*.png` (all figures), `plots/results_table.csv`, `cifar10_cnn_scratch_aug.h5`, `cifar10_mobilenetv2_finetuned.h5`.
**Before submission:** *Restart & Run All* with `FAST_MODE = False` → export to HTML (`File → Download as → HTML`) → push to GitHub with README + video link.